# Step 2.14: the cross-check against NautilusTrader and NinjaTrader 8

This notebook is the check of step 2.14 (D-51, D-59). `just xcheck` ran the same strategies through FEFAT, through NautilusTrader on MESU2026's August 2026, and read NinjaTrader 8's exported trades of the market strategy on MNQU2026, and wrote `docs/validation/nautilus-comparison.md`, where every differing fill has a class. The notebook opens what the run left under `<FEFAT_HOME>/workspaces/default/xcheck/`: each engine's trades beside FEFAT's, the classified pairs, and the gross profit and loss, so that you can look at the trades the report counts. It prints counts, dates, times, and tick differences; the tables it shows hold your data's prices, so its outputs are stripped before a commit, as every notebook's are.

Run `just xcheck` first; without its folder the notebook says so and stops.

In [ ]:
import os
from pathlib import Path

import polars as pl

from fefat.config import resolve_paths

XCHECK = resolve_paths(os.environ).workspace_dir / "xcheck"
MATCH = "match"
pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_width_chars(140)


def utc(instants):
    return pl.from_epoch(instants, time_unit="ns").dt.replace_time_zone("UTC")


def classified_of(comparison):
    folder = XCHECK / comparison / "classified"
    return {path.stem: pl.read_parquet(path) for path in sorted(folder.glob("*.parquet"))}


def counts(classified):
    return (
        classified.group_by("outcome")
        .len("trades")
        .with_columns(share=pl.col("trades") / classified.height)
        .sort("trades", descending=True)
    )


ready = (XCHECK / "nautilus" / "classified").is_dir()
print(f"xcheck folder ready: {ready}")
if not ready:
    print("Run `just xcheck` first: nothing to open.")

## NinjaTrader 8 beside FEFAT (D-59)

The market strategy on MNQU2026 from the store's NinjaTrader import, FEFAT's trades matched with the Strategy Analyzer's export by the decision that entered them and their side. The first table counts the pairs by outcome; the second shows the pairs that differ, with each engine's exit reason and FEFAT's fill prices less NinjaTrader's in ticks; the third the entries, exits, and gross profit and loss of both engines side by side.

Expected: `NinjaTrader 8: 1354 trades matched with FEFAT's 1354`, every outcome `match` or `bar_path` but one `gap_limit_clamped`, and the two engines' gross profit and loss as the report prints them.

In [ ]:
nt8 = classified_of("nt8") if ready else {}
if not nt8:
    print(
        "No NinjaTrader 8 comparison yet: export the Strategy Analyzer's trades (python/tests/external/README.md)."
    )
for classified in nt8.values():
    fefat_trades = pl.read_parquet(XCHECK / "nt8" / "fefat" / "trades_market.parquet")
    print(
        f"NinjaTrader 8: {int(classified['in_other'].sum())} trades matched with FEFAT's "
        f"{int(classified['in_fefat'].sum())}; {fefat_trades.height} FEFAT trades"
    )
    print(counts(classified))
    differing = classified.filter(pl.col("outcome") != MATCH).select(
        utc(pl.col("entry_decision")).alias("decision"),
        "side",
        "outcome",
        "entry_ticks",
        "exit_ticks",
        "note",
    )
    print(differing)

In [ ]:
if nt8:
    import sys

    import fefat as ff

    REPOSITORY = Path(ff.__file__).resolve().parents[4]
    sys.path.insert(0, str(REPOSITORY / "python" / "tests" / "external" / "src"))
    import json

    from fefat_xcheck.nt8_export import read_export

    export = sorted((XCHECK / "nt8" / "export").glob("*.csv"))[-1]
    facts = json.loads((XCHECK / "nt8" / "fefat" / "fefat.json").read_text(encoding="utf-8"))
    theirs = read_export(export, point_value=int(facts["instrument"]["point_value"]))
    ours = pl.read_parquet(XCHECK / "nt8" / "fefat" / "trades_market.parquet")
    side_by_side = (
        ours.select(
            "entry_decision",
            "side",
            utc(pl.col("entry_ts")).alias("fefat_entry"),
            pl.col("entry_price").alias("fefat_entry_price"),
            utc(pl.col("exit_ts")).alias("fefat_exit"),
            pl.col("exit_price").alias("fefat_exit_price"),
            pl.col("exit_reason").alias("fefat_reason"),
            (pl.col("gross_pnl") / 1_000_000).alias("fefat_gross_pnl"),
        )
        .join(
            theirs.select(
                "entry_decision",
                "side",
                utc(pl.col("entry_ts")).alias("nt8_entry"),
                pl.col("entry_price").alias("nt8_entry_price"),
                utc(pl.col("exit_ts")).alias("nt8_exit"),
                pl.col("exit_price").alias("nt8_exit_price"),
                pl.col("exit_reason").alias("nt8_reason"),
                pl.col("gross_pnl").alias("nt8_gross_pnl"),
            ),
            on=["entry_decision", "side"],
            how="full",
            coalesce=True,
        )
        .sort("entry_decision")
    )
    print(
        f"gross profit and loss: FEFAT {side_by_side['fefat_gross_pnl'].sum():,.2f} USD, "
        f"NinjaTrader 8 {side_by_side['nt8_gross_pnl'].sum():,.2f} USD over {side_by_side.height} pairs"
    )
    print(
        side_by_side.filter(pl.col("fefat_reason") != pl.col("nt8_reason")).drop("entry_decision")
    )

## NautilusTrader beside FEFAT

The three strategies on MESU2026 under both of Nautilus's bar paths: the pairs by outcome for each run, as the report's tables count them.

Expected: six runs, each with `unknown` absent from its outcomes; the market strategy's differences `bar_boundary` above all, the limit and stop strategies' `leg_released_through` and `entry_bar_path`.

In [ ]:
for name, classified in (classified_of("nautilus") if ready else {}).items():
    print(f"{name}: {classified.height} pairs")
    print(counts(classified))

## Your check

Read `docs/validation/nautilus-comparison.md`: the market strategy's match rate against NinjaTrader 8 is the highest, since both fill a market order at the next minute's open, and against Nautilus the largest class is `bar_boundary`, Nautilus filling at the deciding bar's close. Pick a differing pair in the tables above and look at its bars in your store with `ff.market.load("MNQU2026", "1m", <date>, <date>, provider="ninjatrader")`: the class's explanation in the report should account for the two exits.